# 01 — Audit, preview and convert TempleRAIL
No dataset downloads, GPU setup, weights, training or Ultralytics. A standard
Colab CPU session with its bundled Pillow is sufficient. Audit and preview
are read-only; nothing is saved until explicit confirmation in cell 9.
Raw XML/TXT/images are never modified. No input annotations.json is needed.


## 1. Repository and Drive
Upload and extract this repository's code to `/content/aquafina-yolo-detector`.
Include src, configs, scripts, requirements and pyproject.toml; no raw data or
weights in the repository ZIP. Mount Drive below. Do not use Run all to approve
conversion: review audit and previews first. Cell numbers include markdown.


In [ ]:
import sys
sys.dont_write_bytecode = True
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/aquafina-yolo-detector')
assert (REPO / 'pyproject.toml').is_file(), 'Extract repository code first'
sys.path.insert(0, str(REPO / 'src'))
TEMPLE_ROOT = Path('/content/drive/MyDrive/aquafina-yolo/raw/temple/extracted/detection_dataset')
PROCESSED_ROOT = Path('/content/drive/MyDrive/aquafina-yolo/processed/temple')
from aquafina_detector.temple import audit_temple, preview_temple, convert_temple
AUDIT = None
PREVIEW_SHOWN = False
CONVERSION_RESULT = None
print('Raw (read-only):', TEMPLE_ROOT)
print('Proposed output (not created):', PROCESSED_ROOT)


## 2. Read-only audit
Decode every image; check image/XML/TXT pairing, dimensions, boxes and classes.
Hash all raw files for immutability checks. This can take several minutes on
Drive. Ignore train.txt for membership: preserve unique val.txt IDs and use
all JPEGImages IDs minus validation. Expect 4,000 train / 870 val / zero overlap.
Known dog and corrupt-train-list warnings do not alone block conversion.
Unexplained class/geometry mismatches and cross-split identical images do block it.


In [ ]:
import json
PREVIEW_SHOWN = False
CONVERSION_RESULT = None
AUDIT = audit_temple(TEMPLE_ROOT)
print(json.dumps(AUDIT.audit, indent=2))
print(json.dumps(AUDIT.class_counts, indent=2))
print(json.dumps(AUDIT.anomalies, indent=2))
print('Audit is in memory only; no report files written.')


## 3. Read-only preview
Green: Aquafina boxes kept. Orange: competitor boxes become background.
Red: XML dog box excluded. Samples prioritize the dog image and available
positive/mixed/negative subsets. Inspect all reported anomalies, not just these
samples. Source annotations do not prove visible brand identity in every image.


In [ ]:
from IPython.display import display
assert AUDIT is not None, 'Run audit cell 5 first'
previews = preview_temple(AUDIT, limit=8)
for caption, image in previews:
    print(caption)
    display(image)
PREVIEW_SHOWN = bool(previews)
print('Preview displayed; nothing saved. Blocking errors:', AUDIT.audit['blocking_errors'])


## 4. Explicit confirmation before the first output write
After reviewing audit and preview, change CONFIRM_CONVERSION to True and run
cell 9. Keep it False to remain read-only. A clean audit and completed preview
are required. Conversion writes exclusively under PROCESSED_ROOT, copies images
(no raw symlinks), and checks raw hashes before/after. Existing output is never
overwritten; use a new child/version path for another conversion.

Aquafina Darknet class 0 becomes model class 0 / COCO category 1. Keep every
image; competitor-only images have empty annotations, and mixed images retain
Aquafina boxes only. Dog objects are excluded. No test split is manufactured.


In [ ]:
CONFIRM_CONVERSION = False
if not CONFIRM_CONVERSION:
    print('Read-only mode: conversion not confirmed; no output written.')
else:
    assert AUDIT is not None and PREVIEW_SHOWN, 'Run audit cell 5 and preview cell 7 first'
    assert AUDIT.audit['blocking_errors'] == 0, 'Resolve blocking audit errors before conversion'
    CONVERSION_RESULT = convert_temple(AUDIT, PROCESSED_ROOT, confirm=True)
    print(json.dumps(CONVERSION_RESULT, indent=2))


## 5. Read back completed outputs
audit.json, anomaly_report.json, class_counts.json, train_manifest.json,
val_manifest.json, split_manifest.json and conversion.json accompany the
annotations/train.json, annotations/val.json and train2017/val2017 images.
The conversion marker is written last; its absence means an incomplete output.
Stop here. Training is a separate future action, not part of this notebook.


In [ ]:
if CONVERSION_RESULT is None:
    print('No conversion performed in this session.')
else:
    from aquafina_detector.common import read_json
    marker = read_json(PROCESSED_ROOT / 'conversion.json')
    print({key: marker[key] for key in ('status', 'train_images', 'val_images', 'raw_sha256_before_after_equal')})
    print('Files:', sorted(path.name for path in PROCESSED_ROOT.iterdir()))
